In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, LabelEncoder
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import balanced_accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


import warnings
warnings.filterwarnings("ignore", message=".*bagging_fraction.*")


#####################################
# Helper: Custom Target Encoder
#####################################
class CustomTargetEncoder(BaseEstimator, TransformerMixin):
    """
    Replaces each category with a smoothed average of the target.
   
    Parameters
    ----------
    cols : list of str
        Columns to encode.
    smoothing : float, default=1.0
        Balances the category's average vs. the global average.
    """
    def __init__(self, cols=None, smoothing=1.0):
        self.cols = cols
        self.smoothing = smoothing
        self.target_means_ = {}
        self.global_mean_ = None


    def fit(self, X, y):
        df = X.copy()
        df['target'] = y  # y should be numeric
        self.global_mean_ = y.mean()
        for col in self.cols:
            stats = df.groupby(col)['target'].agg(['count', 'mean'])
            stats['smoothed'] = ((stats['count'] * stats['mean']) +
                                 (self.smoothing * self.global_mean_)) / (stats['count'] + self.smoothing)
            self.target_means_[col] = stats['smoothed'].to_dict()
        return self


    def transform(self, X):
        X_transformed = X.copy()
        for col in self.cols:
            X_transformed[col] = X_transformed[col].map(self.target_means_[col]).fillna(self.global_mean_)
        return X_transformed


#####################################
# Helper: Age Transformer
#####################################
def parse_age(age_str):
    """
    Convert an age string like "2 years", "5 months", "4 weeks", or "3 days" to number of days.
    """
    try:
        parts = str(age_str).split()
        if len(parts) != 2:
            return np.nan
        value = float(parts[0])
        unit = parts[1].lower()
        if 'year' in unit:
            return value * 365
        elif 'month' in unit:
            return value * 30
        elif 'week' in unit:
            return value * 7
        elif 'day' in unit:
            return value
        else:
            return np.nan
    except Exception:
        return np.nan


def transform_age(X):
    if isinstance(X, pd.DataFrame):
        return X.iloc[:, 0].apply(parse_age).to_numpy().reshape(-1, 1)
    return np.array([parse_age(x) for x in X]).reshape(-1, 1)


age_transformer = FunctionTransformer(func=transform_age, validate=False)


#####################################
# Load and Prepare Data
#####################################
df = pd.read_csv('train.csv')


# For the multiclass problem, we retain all outcome types (5 classes).
df['target'] = df['Outcome Type']  # e.g., outcomes might be: Adoption, Return to Owner, Euthanasia, Died, Transfer


# Use a LabelEncoder to convert outcome strings to numeric labels.
le = LabelEncoder()
y_all = le.fit_transform(df['target'])
print("Label mapping:", dict(zip(le.classes_, range(len(le.classes_)))))


# Define columns to drop (non-useful for modeling)
drop_cols = ['Outcome Type', 'Name', 'Id', 'Intake Time', 'Found Location', 'Outcome Time', 'Date of Birth']


# Prepare features and target.
X = df.drop(columns=drop_cols + ['target'])
y = y_all  # numeric target representing 5 classes


# For clarity, define:
high_card_cols = ['Breed', 'Color']
onehot_cols = ['Intake Type', 'Intake Condition', 'Sex upon Intake', 'Animal Type']


#####################################
# Preprocessor Options
#####################################


# Option 1: Drop "Breed" and "Color"
X_drop = X.drop(columns=high_card_cols)
preprocessor_drop = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), onehot_cols),
        ('age', age_transformer, ['Age upon Intake'])
    ],
    remainder='passthrough'
)


# Option 2: Target Encode "Breed" and "Color"
preprocessor_target = ColumnTransformer(
    transformers=[
        ('target_enc', CustomTargetEncoder(cols=high_card_cols, smoothing=10), high_card_cols),
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), onehot_cols),
        ('age', age_transformer, ['Age upon Intake'])
    ],
    remainder='passthrough'
)


#####################################
# Classifiers and Expanded Parameter Grids
#####################################
classifiers = {
    'LightGBM': {
        'classifier': LGBMClassifier(random_state=42, objective='multiclass', num_class=len(le.classes_),  verbose=0),
        'params': {
            'classifier__n_estimators': [100, 200],
            'classifier__max_depth': [3, 6, 10],
            'classifier__learning_rate': [0.01, 0.1],
            'classifier__num_leaves': [15, 31],
            'classifier__bagging_fraction': [0.8, 1.0]
        }
    },
    # 'LogisticRegression': {
    #     'classifier': LogisticRegression(max_iter=1000, random_state=42),
    #     'params': {
    #         'classifier__C': [0.01, 0.1, 1, 10, 100],
    #         'classifier__penalty': ['l2'],  # l1 or elasticnet not supported with default solver
    #     }
    # },
    # 'KNeighbors': {
    #     'classifier': KNeighborsClassifier(),
    #     'params': {
    #         'classifier__n_neighbors': [3, 5, 7, 9, 11],
    #         'classifier__weights': ['uniform', 'distance'],
    #         'classifier__algorithm': ['auto', 'ball_tree', 'kd_tree']
    #     }
    # },
    # 'Perceptron': {
    #     'classifier': Perceptron(random_state=42),
    #     'params': {
    #         'classifier__eta0': [0.001, 0.01, 0.1],
    #         'classifier__max_iter': [1000, 2000],
    #         'classifier__penalty': ['l2', 'l1', 'elasticnet']
    #     }
    # },
    # 'DecisionTree': {
    #     'classifier': DecisionTreeClassifier(random_state=42),
    #     'params': {
    #         'classifier__max_depth': [None, 5, 10, 20],
    #         'classifier__min_samples_split': [2, 5, 10],
    #         'classifier__min_samples_leaf': [1, 2, 4],
    #         'classifier__criterion': ['gini', 'entropy']
    #     }
    # },
    # 'GaussianNB': {
    #     'classifier': GaussianNB(),
    #     'params': {
    #         'classifier__var_smoothing': np.logspace(-9, -7, 3)
    #     }
    # },
    # 'RandomForest': {
    #     'classifier': RandomForestClassifier(random_state=42),
    #     'params': {
    #         'classifier__n_estimators': [50, 100, 200],
    #         'classifier__max_depth': [5, 10, 20],
    #         'classifier__min_samples_split': [2, 5, 10],
    #         'classifier__min_samples_leaf': [1, 2, 4],
    #         'classifier__max_features': ['sqrt', 'log2']
    #     }
    # },
    # 'XGBoost': {
    #     'classifier': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    #     'params': {
    #         'classifier__n_estimators': [50, 200, 300],
    #         'classifier__max_depth': [3, 6, 10],
    #         'classifier__learning_rate': [0.01, 0.1, 0.3],
    #         'classifier__subsample': [0.8, 1.0]
    #     }
    # },
    # 'Stacking': {
    #     # Use a stacking classifier with a few base learners and a RandomForest as the final estimator.
    #     'classifier': StackingClassifier(
    #         estimators=[
    #             ('lr', LogisticRegression(max_iter=1000, random_state=42)),
    #             ('dt', DecisionTreeClassifier(random_state=42)),
    #             ('knn', KNeighborsClassifier())
    #         ],
    #         final_estimator=RandomForestClassifier(random_state=42),
    #         cv=5,
    #         n_jobs=-1
    #     ),
    #     'params': {
    #         'classifier__final_estimator__n_estimators': [50, 100],
    #         'classifier__final_estimator__max_depth': [None, 5, 10]
    #     }
    # }
}


#####################################
# Nested Cross-Validation Setup
#####################################
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}


for preproc_name, preproc in [('Drop', preprocessor_drop), ('Target', preprocessor_target)]:
    for clf_name, clf_info in classifiers.items():
        pipe = Pipeline([
            ('preprocessor', preproc),
            ('imputer', SimpleImputer(strategy='mean')),
            ('classifier', clf_info['classifier'])
        ])
        grid = GridSearchCV(pipe, clf_info['params'], cv=3, scoring='balanced_accuracy', n_jobs=-1)
        # Use X if preproc option is Target, else use X_drop.
        X_used = X if preproc_name == 'Target' else X_drop
        nested_scores = cross_val_score(grid, X_used, y, cv=outer_cv, scoring='balanced_accuracy', n_jobs=-1, verbose=0)
        mean_score = nested_scores.mean()
        std_score = nested_scores.std()
        results[(preproc_name, clf_name)] = (mean_score, std_score)
        print(f"Preproc: {preproc_name} | Classifier: {clf_name} -> Balanced Accuracy = {mean_score:.4f} (+/- {std_score:.4f})")


#####################################
# Results Summary
#####################################
print("\nNested CV Results Summary:")
for (preproc_name, clf_name), (mean_score, std_score) in results.items():
    print(f"{preproc_name} | {clf_name}: {mean_score:.4f} (+/- {std_score:.4f})")


#####################################
# Optimization Suggestions
#####################################
print("\nOptimization Suggestions:")
print("""
1. Feature Engineering:
   - The high-cardinality features (Breed and Color) might contain useful signals.
     Consider grouping rare categories or using domain knowledge to create aggregated features.
   - Additional features such as day-of-week, seasonality (from Intake Time), or geographic clusters
     (from Found Location) could boost performance.


2. Model Complexity:
   - Ensemble methods (like XGBoost and stacking) tend to capture nonlinear interactions better.
   - Tuning hyperparameters extensively and using ensemble techniques might push your metrics higher.


3. Handling Imbalance:
   - If some classes are underrepresented, consider using class weighting or oversampling techniques
     to balance the training set.


4. Data Quality:
   - Check for noisy or inconsistent entries and consider additional cleaning or advanced imputation methods.
""")





Label mapping: {'Adoption': 0, 'Died': 1, 'Euthanasia': 2, 'Return to Owner': 3, 'Transfer': 4}
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Curr

/root/ML_AustinAnimals/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/root/ML_AustinAnimals/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] No further split

/root/ML_AustinAnimals/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/root/ML_AustinAnimals/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/root/ML_AustinAnimals/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/root/ML_AustinAnimals/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/root/ML_AustinAnimals/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [ ]:
print("\nNested CV Results Summary:")
for (preproc_name, clf_name), (mean_score, std_score) in results.items():
    print(f"{preproc_name} | {clf_name}: {mean_score:.4f} (+/- {std_score:.4f})")
